# MusicBrainz Enrichment Notebook (Genius Only)

This notebook enriches the processed MusicBrainz dataset using **Genius only**.

It adds:
- Genius match metadata (`song_id`, `title`, `artist`, `url`, `match score`)

It is built for large datasets with chunked processing and checkpoint resume.

## 1) Setup

- Input dataset: `../data/processed/musicbrainz/mb_likely_french_stage2_with_genre_long.tsv`
- Output dataset: `../data/processed/musicbrainz/mb_likely_french_stage2_enriched_genius.csv`
- Token source: `../.env` (`GENIUS_ACCESS_TOKEN`)

In [1]:
from __future__ import annotations

import re
import time
from pathlib import Path
from difflib import SequenceMatcher

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm
from bs4 import BeautifulSoup

tqdm.pandas()

In [2]:
BASE_DIR = Path.cwd()
DATA_IN = (BASE_DIR / "../data/processed/musicbrainz/mb_likely_french_stage2_with_genre_long.tsv").resolve()
OUT_DIR = (BASE_DIR / "../data/processed/musicbrainz").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "mb_likely_french_stage2_enriched_genius.csv"
CHECKPOINT_FILE = OUT_DIR / "mb_likely_french_stage2_genius_checkpoint.txt"
ENV_FILE = (BASE_DIR / "../.env").resolve()

CHUNK_SIZE = 500
MAX_ROWS = None
SLEEP_BETWEEN_CALLS = 0.20

print("Input:", DATA_IN)
print("Output:", OUT_FILE)
print("Checkpoint:", CHECKPOINT_FILE)
print("Env file:", ENV_FILE)

Input: /Users/danvoloshin/Desktop/masters/thesis_french_song_nlp/data/processed/musicbrainz/mb_likely_french_stage2_with_genre_long.tsv
Output: /Users/danvoloshin/Desktop/masters/thesis_french_song_nlp/data/processed/musicbrainz/mb_likely_french_stage2_enriched_genius.csv
Checkpoint: /Users/danvoloshin/Desktop/masters/thesis_french_song_nlp/data/processed/musicbrainz/mb_likely_french_stage2_genius_checkpoint.txt
Env file: /Users/danvoloshin/Desktop/masters/thesis_french_song_nlp/.env


## 2) Load Genius token from `.env`

In [3]:
def read_env_value(path: Path, key: str) -> str:
    if not path.exists():
        return ""
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        if k.strip() == key:
            return v.strip().strip('"').strip("'")
    return ""

GENIUS_ACCESS_TOKEN = read_env_value(ENV_FILE, "GENIUS_ACCESS_TOKEN")
if not GENIUS_ACCESS_TOKEN:
    raise ValueError("GENIUS_ACCESS_TOKEN is missing in .env")

print("Genius token loaded:", bool(GENIUS_ACCESS_TOKEN))

Genius token loaded: True


## 3) Resilient HTTP session

In [4]:
def build_session() -> requests.Session:
    retry = Retry(
        total=5,
        read=5,
        connect=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    sess = requests.Session()
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)
    sess.headers.update({"User-Agent": "masters-thesis-genius-enrichment/1.0"})
    return sess

session = build_session()

## 4) Matching helpers + Genius search

In [ ]:
def norm(s: str | None) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def text_similarity(a: str | None, b: str | None) -> float:
    return SequenceMatcher(None, norm(a), norm(b)).ratio()


def safe_get_json(url: str, params: dict | None = None, headers: dict | None = None, timeout: int = 20):
    try:
        resp = session.get(url, params=params, headers=headers, timeout=timeout)
        if resp.status_code == 200:
            return resp.json()
    except requests.RequestException:
        return None
    return None


def safe_get_text(url: str, headers: dict | None = None, timeout: int = 20) -> str | None:
    try:
        resp = session.get(url, headers=headers, timeout=timeout)
        if resp.status_code == 200:
            return resp.text
    except requests.RequestException:
        return None
    return None


def extract_genius_lyrics_from_url(genius_url: str | None) -> str | None:
    if not genius_url:
        return None

    html = safe_get_text(genius_url)
    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")
    blocks = soup.select('div[data-lyrics-container="true"]')
    if not blocks:
        return None

    parts = []
    for block in blocks:
        text = block.get_text("\n", strip=True)
        if text:
            parts.append(text)

    lyrics = "\n\n".join(parts).strip()
    return lyrics or None

def fetch_genius_hit(artist_name: str, song_title: str, token: str) -> dict:
    query = f"{artist_name or ''} {song_title or ''}".strip()
    url = "https://api.genius.com/search"
    headers = {"Authorization": f"Bearer {token}"}
    params = {"q": query}

    payload = safe_get_json(url, params=params, headers=headers)
    if not payload or not isinstance(payload, dict):
        return {
            "genius_found": False,
            "genius_song_id": None,
            "genius_title": None,
            "genius_artist": None,
            "genius_url": None,
            "genius_match_score": 0.0,
            "lyrics_found": False,
            "lyrics_text": None,
            "lyrics_source": None,
        }

    hits = (((payload.get("response") or {}).get("hits")) or [])
    if not hits:
        return {
            "genius_found": False,
            "genius_song_id": None,
            "genius_title": None,
            "genius_artist": None,
            "genius_url": None,
            "genius_match_score": 0.0,
            "lyrics_found": False,
            "lyrics_text": None,
            "lyrics_source": None,
        }

    def score(hit: dict) -> float:
        result = hit.get("result") or {}
        s1 = text_similarity(song_title, result.get("title"))
        s2 = text_similarity(artist_name, (result.get("primary_artist") or {}).get("name"))
        return 0.65 * s1 + 0.35 * s2

    best = max(hits, key=score)
    result = best.get("result") or {}
    best_score = round(float(score(best)), 4)
    genius_url = result.get("url")

    lyrics_text = None
    if best_score >= 0.75 and genius_url:
        lyrics_text = extract_genius_lyrics_from_url(genius_url)

    return {
        "genius_found": True,
        "genius_song_id": result.get("id"),
        "genius_title": result.get("title"),
        "genius_artist": (result.get("primary_artist") or {}).get("name"),
        "genius_url": genius_url,
        "genius_match_score": best_score,
        "lyrics_found": bool(lyrics_text),
        "lyrics_text": lyrics_text,
        "lyrics_source": "genius" if lyrics_text else None,
    }

## 5) Row-level enrichment

In [6]:
def enrich_row(row: pd.Series) -> dict:
    title = row.get("song_title", "")
    artist = row.get("artist_name") or row.get("artist_credit_name") or ""

    out = fetch_genius_hit(artist, title, token=GENIUS_ACCESS_TOKEN)
    time.sleep(SLEEP_BETWEEN_CALLS)
    return out

## 6) Select columns + dry run

In [7]:
base_cols = [
    "track_id",
    "recording_id",
    "release_id",
    "artist_id",
    "year",
    "song_title",
    "artist_credit_name",
    "artist_name",
    "language_name",
    "iso_code_3t",
    "release_country_code",
    "artist_country_code",
    "genre_name",
    "likely_french_score",
]

source_cols = pd.read_csv(DATA_IN, sep="	", nrows=0).columns.tolist()
missing = [c for c in base_cols if c not in source_cols]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

print("Selected columns:", base_cols)

Selected columns: ['track_id', 'recording_id', 'release_id', 'artist_id', 'year', 'song_title', 'artist_credit_name', 'artist_name', 'language_name', 'iso_code_3t', 'release_country_code', 'artist_country_code', 'genre_name', 'likely_french_score']


In [ ]:
DRY_RUN_N = 1

dry_df = pd.read_csv(DATA_IN, sep="	", usecols=base_cols, nrows=DRY_RUN_N)
dry_enriched = pd.concat(
    [dry_df, dry_df.progress_apply(enrich_row, axis=1, result_type="expand")],
    axis=1,
)

display(dry_enriched.head(10))

## 7) Full chunked run (resumable)

In [ ]:
def read_checkpoint(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        return int(path.read_text().strip())
    except Exception:
        return 0


def write_checkpoint(path: Path, value: int) -> None:
    path.write_text(str(value))


start_row = read_checkpoint(CHECKPOINT_FILE)
print("Starting from row:", start_row)

processed_rows = 0
written_once = OUT_FILE.exists() and OUT_FILE.stat().st_size > 0

reader = pd.read_csv(
    DATA_IN,
    sep="	",
    usecols=base_cols,
    chunksize=CHUNK_SIZE,
)

for chunk_i, chunk in enumerate(reader):
    chunk_start = chunk_i * CHUNK_SIZE
    chunk_end = chunk_start + len(chunk)

    if chunk_end <= start_row:
        continue

    if chunk_start < start_row < chunk_end:
        keep_from = start_row - chunk_start
        chunk = chunk.iloc[keep_from:].copy()

    if MAX_ROWS is not None and processed_rows >= MAX_ROWS:
        break

    if MAX_ROWS is not None:
        remaining = MAX_ROWS - processed_rows
        if remaining < len(chunk):
            chunk = chunk.iloc[:remaining].copy()

    enriched_part = chunk.progress_apply(enrich_row, axis=1, result_type="expand")
    out_chunk = pd.concat([chunk.reset_index(drop=True), enriched_part.reset_index(drop=True)], axis=1)

    out_chunk.to_csv(OUT_FILE, mode="a", index=False, header=not written_once)
    written_once = True

    processed_rows += len(out_chunk)
    write_checkpoint(CHECKPOINT_FILE, chunk_start + len(chunk))
    print(f"Chunk {chunk_i}: wrote {len(out_chunk)} rows | total this run={processed_rows}")

print("Done.")
print("Output file:", OUT_FILE)
print("Checkpoint row:", read_checkpoint(CHECKPOINT_FILE))

### 7.1) Checkpoint Utilities

Define checkpoint read/write helpers used to make long enrichment runs resumable and safe to restart.

In [ ]:
def read_checkpoint(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        return int(path.read_text().strip())
    except Exception:
        return 0


def write_checkpoint(path: Path, value: int) -> None:
    path.write_text(str(value))


start_row = read_checkpoint(CHECKPOINT_FILE)
print("Starting from row:", start_row)

processed_rows = 0
written_once = OUT_FILE.exists() and OUT_FILE.stat().st_size > 0

# Keep already enriched rows
DEDUP_KEY = "recording_id"
existing_keys = set()
if OUT_FILE.exists():
    prev = pd.read_csv(OUT_FILE, usecols=[DEDUP_KEY])
    existing_keys = set(prev[DEDUP_KEY].dropna().astype(str))
print("Existing enriched keys:", len(existing_keys))

reader = pd.read_csv(
    DATA_IN,
    sep="\t",
    usecols=base_cols,
    chunksize=CHUNK_SIZE,
)

for chunk_i, chunk in enumerate(reader):
    chunk_start = chunk_i * CHUNK_SIZE
    chunk_end = chunk_start + len(chunk)

    if chunk_end <= start_row:
        continue

    if chunk_start < start_row < chunk_end:
        keep_from = start_row - chunk_start
        chunk = chunk.iloc[keep_from:].copy()

    if MAX_ROWS is not None and processed_rows >= MAX_ROWS:
        break

    if MAX_ROWS is not None:
        remaining = MAX_ROWS - processed_rows
        if remaining < len(chunk):
            chunk = chunk.iloc[:remaining].copy()

    chunk["year_num"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk = chunk[chunk["year_num"].between(1925, 2025, inclusive="both")].copy()
    chunk = chunk.drop(columns=["year_num"])

    chunk["_k"] = chunk[DEDUP_KEY].astype(str)
    chunk = chunk.drop_duplicates(subset=["_k"], keep="first")
    chunk = chunk[~chunk["_k"].isin(existing_keys)].copy()

    if chunk.empty:
        write_checkpoint(CHECKPOINT_FILE, chunk_end)
        print(f"Chunk {chunk_i}: nothing to write after filter/dedup")
        continue

    enriched_part = chunk.progress_apply(enrich_row, axis=1, result_type="expand")
    out_chunk = pd.concat(
        [chunk.drop(columns=["_k"]).reset_index(drop=True), enriched_part.reset_index(drop=True)],
        axis=1,
    )

    out_chunk.to_csv(OUT_FILE, mode="a", index=False, header=not written_once)
    written_once = True

    processed_rows += len(out_chunk)
    existing_keys.update(out_chunk[DEDUP_KEY].dropna().astype(str))

    write_checkpoint(CHECKPOINT_FILE, chunk_end)
    print(f"Chunk {chunk_i}: wrote {len(out_chunk)} rows | total this run={processed_rows}")

print("Done.")
print("Output file:", OUT_FILE)
print("Checkpoint row:", read_checkpoint(CHECKPOINT_FILE))

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import random
import requests
import pandas as pd
import time

MAX_WORKERS = 6
SLEEP_BETWEEN_CALLS = 0.05
DEDUP_KEY = "recording_id"

def safe_get_json(url: str, params: dict | None = None, headers: dict | None = None, timeout: int = 20):
    max_retries = 4
    base_backoff = 1
    for attempt in range(max_retries):
        try:
            resp = session.get(url, params=params, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                return resp.json()

            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                wait = float(retry_after) if retry_after else (base_backoff * (2 ** attempt))
                time.sleep(wait + random.uniform(0, 0.2))
                continue

            if resp.status_code in (500, 502, 503, 504):
                time.sleep(base_backoff * (2 ** attempt) + random.uniform(0, 0.2))
                continue

            return None
        except requests.RequestException:
            time.sleep(base_backoff * (2 ** attempt) + random.uniform(0, 0.2))
    return None


def safe_get_text(url: str, headers: dict | None = None, timeout: int = 20) -> str | None:
    max_retries = 6
    base_backoff = 0.8
    for attempt in range(max_retries):
        try:
            resp = session.get(url, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                return resp.text

            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                wait = float(retry_after) if retry_after else (base_backoff * (2 ** attempt))
                time.sleep(wait + random.uniform(0, 0.2))
                continue

            if resp.status_code in (500, 502, 503, 504):
                time.sleep(base_backoff * (2 ** attempt) + random.uniform(0, 0.2))
                continue

            return None
        except requests.RequestException:
            time.sleep(base_backoff * (2 ** attempt) + random.uniform(0, 0.2))
    return None

# checkpoint helpers
def read_checkpoint(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        return int(path.read_text().strip())
    except Exception:
        return 0


def write_checkpoint(path: Path, value: int) -> None:
    path.write_text(str(value))

# worker
def enrich_row_worker(row_dict: dict) -> dict:
    global session
    old_session = session
    session = build_session()
    try:
        out = enrich_row(pd.Series(row_dict))
        if SLEEP_BETWEEN_CALLS > 0:
            time.sleep(SLEEP_BETWEEN_CALLS)
        return out
    finally:
        session = old_session

start_row = read_checkpoint(CHECKPOINT_FILE)
print("Starting from raw row:", start_row)

processed_rows = 0
written_once = OUT_FILE.exists() and OUT_FILE.stat().st_size > 0

existing_keys = set()
if OUT_FILE.exists():
    prev = pd.read_csv(OUT_FILE, usecols=[DEDUP_KEY])
    existing_keys = set(prev[DEDUP_KEY].dropna().astype(str))
print("Existing enriched keys:", len(existing_keys))

reader = pd.read_csv(
    DATA_IN,
    sep="\t",
    usecols=base_cols,
    chunksize=CHUNK_SIZE,
)

for chunk_i, raw_chunk in enumerate(reader):
    chunk_start = chunk_i * CHUNK_SIZE
    chunk_end = chunk_start + len(raw_chunk)

    if chunk_end <= start_row:
        continue

    chunk = raw_chunk.copy()
    keep_from = 0
    if chunk_start < start_row < chunk_end:
        keep_from = start_row - chunk_start
        chunk = chunk.iloc[keep_from:].copy()

    if MAX_ROWS is not None and processed_rows >= MAX_ROWS:
        break

    raw_processed_until = chunk_end
    if MAX_ROWS is not None:
        remaining = MAX_ROWS - processed_rows
        if remaining < len(chunk):
            chunk = chunk.iloc[:remaining].copy()
            raw_processed_until = chunk_start + keep_from + remaining

    chunk["year_num"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk = chunk[chunk["year_num"].between(1925, 2025, inclusive="both")].copy()
    chunk = chunk.drop(columns=["year_num"])

    chunk["_k"] = chunk[DEDUP_KEY].astype(str)
    chunk = chunk.drop_duplicates(subset=["_k"], keep="first")
    chunk = chunk[~chunk["_k"].isin(existing_keys)].copy()

    if chunk.empty:
        write_checkpoint(CHECKPOINT_FILE, raw_processed_until)
        print(f"Chunk {chunk_i}: nothing to write after filter/dedup")
        continue

    records = chunk.to_dict("records")
    results = [None] * len(records)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        fut_to_idx = {ex.submit(enrich_row_worker, rec): i for i, rec in enumerate(records)}
        for fut in tqdm(as_completed(fut_to_idx), total=len(fut_to_idx), desc=f"Chunk {chunk_i}"):
            i = fut_to_idx[fut]
            try:
                results[i] = fut.result()
            except Exception:
                results[i] = {
                    "genius_found": False,
                    "genius_song_id": None,
                    "genius_title": None,
                    "genius_artist": None,
                    "genius_url": None,
                    "genius_match_score": 0.0,
                    "lyrics_found": False,
                    "lyrics_text": None,
                    "lyrics_source": None,
                }

    enriched_part = pd.DataFrame(results)
    out_chunk = pd.concat(
        [chunk.drop(columns=["_k"]).reset_index(drop=True), enriched_part.reset_index(drop=True)],
        axis=1,
    )

    out_chunk.to_csv(OUT_FILE, mode="a", index=False, header=not written_once)
    written_once = True
    processed_rows += len(out_chunk)
    existing_keys.update(out_chunk[DEDUP_KEY].dropna().astype(str))

    write_checkpoint(CHECKPOINT_FILE, raw_processed_until)
    print(f"Chunk {chunk_i}: wrote {len(out_chunk)} rows | total this run={processed_rows}")

print("Done.")
print("Output file:", OUT_FILE)
print("Checkpoint row:", read_checkpoint(CHECKPOINT_FILE))

Starting from raw row: 582000
Existing enriched keys: 288622


Chunk 1164:   0%|          | 0/212 [00:00<?, ?it/s]

Chunk 1164: wrote 212 rows | total this run=212


Chunk 1165:   0%|          | 0/384 [00:00<?, ?it/s]

Chunk 1165: wrote 384 rows | total this run=596


...

Chunk 1210:   0%|          | 0/262 [00:00<?, ?it/s]

Chunk 1210: wrote 262 rows | total this run=9736


Chunk 1211:   0%|          | 0/178 [00:00<?, ?it/s]

Chunk 1211: wrote 178 rows | total this run=9914


Chunk 1212:   0%|          | 0/322 [00:00<?, ?it/s]

In [8]:
import pandas as pd

df = pd.read_csv(OUT_FILE, usecols=["year", "lyrics_text"])

df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"]).copy()
df["year"] = df["year"].astype(int)

df["has_lyrics"] = (
    df["lyrics_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

year_stats = (
    df.groupby("year", as_index=False)
      .agg(
          rows_total=("has_lyrics", "size"),
          lyrics_filled=("has_lyrics", "sum"),
      )
      .sort_values("year")
)

year_stats["lyrics_fill_rate"] = (year_stats["lyrics_filled"] / year_stats["rows_total"]).round(4)

# Force full print (no "...")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print(year_stats.to_string(index=False))
print("\nTotal rows:", len(df))
print("Rows with lyrics:", int(df["has_lyrics"].sum()))

 year  rows_total  lyrics_filled  lyrics_fill_rate
 1929           6              0            0.0000
 1930          14              0            0.0000
 1931           9              0            0.0000
 1932           6              0            0.0000
 1933           2              1            0.5000
 1934           4              0            0.0000
 1935          11              0            0.0000
 1936           9              2            0.2222
 1937           7              0            0.0000
 1938           2              0            0.0000
 1939           2              0            0.0000
 1941           3              0            0.0000
 1942           4              0            0.0000
 1944           2              0            0.0000
 1945           1              0            0.0000
 1947           5              2            0.4000
 1948           1              0            0.0000
 1949           3              0            0.0000
 1950           5              

### 7.2) Parallel Enrichment Execution

Run threaded enrichment across chunks, persist progress, and maintain stable output writes.

In [ ]:
# 1) Audit: source vs enriched coverage by year
src = pd.read_csv(DATA_IN, sep="\t", usecols=["recording_id", "year"])
src["year"] = pd.to_numeric(src["year"], errors="coerce")
src = src.dropna(subset=["year"]).copy()
src["year"] = src["year"].astype(int)
src = src[src["year"].between(1925, 2025)]

out = pd.read_csv(OUT_FILE, usecols=["recording_id", "year", "lyrics_text"])
out["year"] = pd.to_numeric(out["year"], errors="coerce")
out = out.dropna(subset=["year"]).copy()
out["year"] = out["year"].astype(int)

src_year = src.groupby("year", as_index=False).agg(src_rows=("recording_id", "size"))
out_year = out.groupby("year", as_index=False).agg(out_rows=("recording_id", "size"))

audit = src_year.merge(out_year, on="year", how="left").fillna(0)
audit["out_rows"] = audit["out_rows"].astype(int)
audit["missing_rows"] = audit["src_rows"] - audit["out_rows"]

print(audit[audit["year"].between(1925, 1932)].to_string(index=False))

 year  src_rows  out_rows  missing_rows
 1925         6         0             6
 1926         6         0             6
 1927        21         0            21
 1928        21         0            21
 1929        24         0            24
 1930        47         0            47
 1931        40         0            40
 1932        35         0            35


### 7.3) Coverage Audit and Backfill Passes

Inspect enrichment coverage and run targeted historical backfills for under-covered periods.

In [20]:
# 2) Backfill 1925-1932 only, append safely to a separate file (no checkpoint changes)
from pathlib import Path

BACKFILL_OUT = OUT_FILE.with_name("mb_likely_french_stage2_enriched_genius_backfill_1925_1932.csv")

# Existing keys already enriched
existing_keys = set(pd.read_csv(OUT_FILE, usecols=["recording_id"])["recording_id"].dropna().astype(str))

# Pull source candidates
base = pd.read_csv(DATA_IN, sep="\t", usecols=base_cols)
base["year"] = pd.to_numeric(base["year"], errors="coerce")
cand = base[
    base["year"].between(1925, 1932, inclusive="both")
].copy()

cand["_k"] = cand["recording_id"].astype(str)
cand = cand.drop_duplicates(subset=["_k"], keep="first")
cand = cand[~cand["_k"].isin(existing_keys)].copy()

print("Backfill candidates:", len(cand))

if len(cand):
    enriched = cand.progress_apply(enrich_row, axis=1, result_type="expand")
    out_backfill = pd.concat([cand.drop(columns=["_k"]).reset_index(drop=True), enriched.reset_index(drop=True)], axis=1)
    out_backfill.to_csv(BACKFILL_OUT, index=False)
    print("Wrote:", BACKFILL_OUT)

Backfill candidates: 182


  0%|          | 0/182 [00:00<?, ?it/s]

Wrote: /Users/danvoloshin/Desktop/masters/thesis_french_song_nlp/data/processed/musicbrainz/mb_likely_french_stage2_enriched_genius_backfill_1925_1932.csv


In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

MAX_WORKERS = 6

def clean_title(t: str) -> str:
    t = str(t or "")
    t = re.sub(r"\s*\(.*?\)\s*", " ", t)
    t = re.sub(r"\s*\[.*?\]\s*", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def fetch_genius_hit_with_threshold(artist_name: str, song_title: str, token: str, threshold: float = 0.60) -> dict:
    queries = [
        (artist_name or "", song_title or ""),
        (artist_name or "", clean_title(song_title)),
    ]

    best = None
    for a, t in queries:
        hit = fetch_genius_hit(a, t, token)
        if (best is None) or (hit.get("genius_match_score", 0) > best.get("genius_match_score", 0)):
            best = hit

    if not best:
        return {
            "genius_found": False, "genius_song_id": None, "genius_title": None, "genius_artist": None,
            "genius_url": None, "genius_match_score": 0.0, "lyrics_found": False, "lyrics_text": None, "lyrics_source": None
        }

    # force no lyrics when below threshold
    if best.get("genius_match_score", 0) < threshold:
        best["lyrics_found"] = False
        best["lyrics_text"] = None
        best["lyrics_source"] = None

    return best

def enrich_row_boost_parallel(row_dict: dict, threshold: float = 0.60) -> dict:
    title = row_dict.get("song_title", "")
    artist = row_dict.get("artist_name") or row_dict.get("artist_credit_name") or ""
    try:
        return fetch_genius_hit_with_threshold(artist, title, GENIUS_ACCESS_TOKEN, threshold=threshold)
    except Exception:
        return {
            "genius_found": False, "genius_song_id": None, "genius_title": None, "genius_artist": None,
            "genius_url": None, "genius_match_score": 0.0, "lyrics_found": False, "lyrics_text": None, "lyrics_source": None
        }

def run_boost_parallel(df_retry: pd.DataFrame, threshold: float = 0.60, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    records = df_retry.to_dict("records")
    results = [None] * len(records)

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        fut_to_idx = {
            ex.submit(enrich_row_boost_parallel, rec, threshold): i
            for i, rec in enumerate(records)
        }
        for fut in tqdm(as_completed(fut_to_idx), total=len(fut_to_idx), desc=f"Boost pre-1990 ({max_workers} workers)"):
            i = fut_to_idx[fut]
            results[i] = fut.result()

    return pd.DataFrame(results)


In [ ]:
BOOST_OUT = OUT_FILE.with_name("mb_likely_french_stage2_enriched_genius_boost_pre1990.csv")
BOOST_CHECKPOINT = OUT_FILE.with_name("mb_likely_french_stage2_enriched_genius_boost_pre1990.checkpoint.txt")

BATCH_SIZE = 300
THRESHOLD = 0.50
MAX_WORKERS = 4

def read_ckpt(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        return int(path.read_text().strip())
    except Exception:
        return 0

def write_ckpt(path: Path, value: int) -> None:
    path.write_text(str(value))

full = pd.read_csv(
    OUT_FILE,
    usecols=["recording_id", "year", "song_title", "artist_name", "artist_credit_name", "lyrics_text"]
)
full["year"] = pd.to_numeric(full["year"], errors="coerce")
full = full.dropna(subset=["year"]).copy()
full["year"] = full["year"].astype(int)

retry = full[
    (full["year"] < 1990) &
    (full["lyrics_text"].fillna("").astype(str).str.strip() == "")
].copy()

retry = retry.drop_duplicates(subset=["recording_id"], keep="first").reset_index(drop=True)

# Skip rows already written to BOOST_OUT
done_ids = set()
if BOOST_OUT.exists():
    done_ids = set(pd.read_csv(BOOST_OUT, usecols=["recording_id"])["recording_id"].dropna().astype(str))

retry["_rid"] = retry["recording_id"].astype(str)
retry = retry[~retry["_rid"].isin(done_ids)].drop(columns=["_rid"]).reset_index(drop=True)

start_idx = read_ckpt(BOOST_CHECKPOINT)
print("Retry candidates:", len(retry))
print("Resume from index:", start_idx)

written_once = BOOST_OUT.exists() and BOOST_OUT.stat().st_size > 0

for batch_start in range(start_idx, len(retry), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(retry))
    batch = retry.iloc[batch_start:batch_end].copy()

    # parallel enrichment from your safeguarded function
    boosted = run_boost_parallel(batch, threshold=THRESHOLD, max_workers=MAX_WORKERS)

    boosted_out = pd.concat([batch.reset_index(drop=True), boosted.reset_index(drop=True)], axis=1)
    boosted_out = boosted_out[boosted_out["lyrics_found"] == True].copy()

    if not boosted_out.empty:
        boosted_out.to_csv(BOOST_OUT, mode="a", index=False, header=not written_once)
        written_once = True

    # checkpoint always advances by retry index, independent of filtered writes
    write_ckpt(BOOST_CHECKPOINT, batch_end)

    print(
        f"Batch {batch_start}:{batch_end} | input={len(batch)} | "
        f"lyrics_found={len(boosted_out)} | checkpoint={batch_end}"
    )

print("Done.")
print("BOOST_OUT:", BOOST_OUT)
print("BOOST_CHECKPOINT:", BOOST_CHECKPOINT, "->", read_ckpt(BOOST_CHECKPOINT))

Retry candidates: 6317
Resume from index: 3900


Boost pre-1990 (4 workers):   0%|          | 0/300 [00:00<?, ?it/s]

Batch 3900:4200 | input=300 | lyrics_found=6 | checkpoint=4200


Boost pre-1990 (4 workers):   0%|          | 0/300 [00:00<?, ?it/s]

Batch 4200:4500 | input=300 | lyrics_found=0 | checkpoint=4500


...

Batch 5700:6000 | input=300 | lyrics_found=5 | checkpoint=6000


Boost pre-1990 (4 workers):   0%|          | 0/300 [00:00<?, ?it/s]

Batch 6000:6300 | input=300 | lyrics_found=4 | checkpoint=6300


Boost pre-1990 (4 workers):   0%|          | 0/17 [00:00<?, ?it/s]

Batch 6300:6317 | input=17 | lyrics_found=0 | checkpoint=6317
Done.
BOOST_OUT: /Users/danvoloshin/Desktop/masters/thesis_french_song_nlp/data/processed/musicbrainz/mb_likely_french_stage2_enriched_genius_boost_pre1990.csv
BOOST_CHECKPOINT: /Users/danvoloshin/Desktop/masters/thesis_french_song_nlp/data/processed/musicbrainz/mb_likely_french_stage2_enriched_genius_boost_pre1990.checkpoint.txt -> 6317


In [ ]:
MERGED_OUT = OUT_FILE.with_name("mb_likely_french_stage2_enriched_genius_merged.csv")

main_df = pd.read_csv(OUT_FILE)
parts = [main_df]

for extra in [BACKFILL_OUT, BOOST_OUT]:
    if Path(extra).exists():
        parts.append(pd.read_csv(extra))

m = pd.concat(parts, ignore_index=True)

m["lyrics_len"] = m["lyrics_text"].fillna("").astype(str).str.len()
m = m.sort_values(["recording_id", "lyrics_len"], ascending=[True, False])
m = m.drop_duplicates(subset=["recording_id"], keep="first").drop(columns=["lyrics_len"])

m.to_csv(MERGED_OUT, index=False)
print("Merged:", MERGED_OUT, "| rows:", len(m), "| lyrics:", (m["lyrics_text"].fillna("").astype(str).str.strip() != "").sum())

## 8) Quick QA checks

In [23]:
preview = pd.read_csv(OUT_FILE, nrows=2000)

summary = {
    "rows_previewed": len(preview),
    "genius_found_rate": float(preview["genius_found"].mean()) if "genius_found" in preview else None,
    "lyrics_found_rate": float(preview["lyrics_found"].mean()) if "lyrics_found" in preview else None,
    "avg_genius_match_score": float(preview["genius_match_score"].fillna(0).mean()) if "genius_match_score" in preview else None,
}

print(summary)
display(preview[[
    "song_title", "artist_name", "genius_title", "genius_artist", "genius_match_score", "lyrics_found", "lyrics_text"
]].head(20))


{'rows_previewed': 2000, 'genius_found_rate': 0.662, 'lyrics_found_rate': 0.63, 'avg_genius_match_score': 0.4368419}


,song_title,artist_name,genius_title,genius_artist,genius_match_score,lyrics_found,lyrics_text
0,Bambino,Dalida,Bambino,Dalida,1.0000,True,"6 Contributors\nTranslations\nDeutsch\nEnglish\nBambino Lyrics\n[Paroles de ""Bambino""]\n[Intro]\nBambino, bambino\nNe pleure pas, Bambino\n[Couplet 1]\nLes yeux battus, la mine triste et les joues blêmes\nTu ne dors plus, tu n’es que l’ombre de toi-même\nSeul dans la rue, tu rôdes comme une âme en peine\nEt tous les soirs, sous sa fenêtre, on peut te voir\n[Refrain]\nJe sais bien que tu l’adores\n(\nBambino, bambino\n)\nEt qu’elle a de jolis yeux\n(\nBambino, bambino\n)\nMais tu es trop jeune encore\n(\nBambino, bambino\n)\nPour jouer les amoureux\n[Couplet 2]\nE gratta, gratta sul tuo mandolino, mon petit Bambino\nTa musique est plus jolie, que tout le ciel de l’Italie\nEt canta, canta de ta voix câline, mon petit Bambino\nTu peux chanter tant que tu veux, elle ne te prend pas au sérieux\n\n[Refrain]\nAvec tes cheveux si blonds\n(\nBambino, bambino\n)\nTu as l’air d’un chérubin\n(\nBambino, bambino\n)\nVa plutôt jouer au ballon\n(\nBambino, bambino\n)\nComme font tous les gamins\n[Couplet 3]\nTu peux fumer comme un Monsieur des cigarettes\nTe déhancher sur le trottoir quand tu la guettes\nTu peux pencher sur ton oreille ta casquette\nCe n’est pas ça que dans son cœur, te vieillira\n[Refrain]\nL’amour et la jalousie\n(\nBambino, bambino\n)\nNe sont pas des jeux d’enfant\n(\nBambino, bambino\n)\nEt tu as toute la vie\n(\nBambino, bambino\n)\nPour souffrir comme les grands\n(\nBambino, bambino\n)\n[Couplet 2]\nE gratta, gratta sul tuo mandolino, mon petit Bambino\nTa musique est plus jolie, que tout le ciel de l’Italie\nEt canta, canta de ta voix câline, mon petit Bambino\nTu peux chanter tant que tu veux, elle ne te prend pas au sérieux\n\n[Refrain]\nSi tu as trop de tourments\n(\nBambino, bambino\n)\nNe les garde pas pour toi\n(\nBambino, bambino\n)\nVa les dire à ta maman\n(\nBambino, bambino\n)\nLes mamans c'est fait pour ça\n[Outro]\nEt là, blotti dans l’ombre douce de ses bras\nPleure un bon coup, et ton chagrin s’envolera"
1,Ciao ciao bambina,Dalida,"Ciao, ciao Bambina",Dalida,0.9814,True,"3 Contributors\nCiao, ciao Bambina Lyrics\nMille violons chantent leur mélodie\nUn arc-en-ciel dans le ciel se déplie\nMais il me semble être encore sous la pluie\nSi malheureuse quand il m'a dit\nTchao Tchao Bambina\nDis-moi je t'aime\nPour la dernière, dernière fois\nBientôt petite\nJe vais te perdre\nJ'ai de la peine\nEmbrasse-moi\nTchao Tchao Bambina\nUn jour on s'aime\nEt l'on se quitte\nL'amour c'est ça...\nDans tes yeux tristes\nTout le ciel pleure\nEt moi je pleure\nPleure avec toi!\nTchao Tchao Bambina\nDis-moi je t'aime\nPour la dernière, dernière fois\nBientôt petite\nJe vais te perdre\nJ'ai de la peine\nEmbrasse-moi\nTchao Tchao Bambina\nQui sait, ma route\nCrois'ra la tienne\nPeut-être un jour\nMais le ciel même\nCe soir en douce\nIl pleure pleure\nSur notre amour!"
2,Les Gitans,Dalida,Les gitans,Dalida,1.0000,True,"2 Contributors\nLes gitans Lyrics\nD'où viens-tu gitan?\nJe viens de Bohème\nD'où viens-tu gitan?\nJe viens d'Italie\nEt toi, beau gitan?\nDe l'Andalousie\nEt toi, vieux gitan, d'où viens-tu?\nJe viens d'un pays qui n'existe plus...\nLes chevaux rassemblés le long de la barrière\nLe flanc gris de poussière\nLe naseau écumant\nLes gitans sont assis près de la flamme claire\nQui jette à la clairière\nLeurs ombres de géants\nEt dans la nuit monte un refrain bizarre\nEt dans la nuit bat le cœur des guitares\nC'est le chant des errants qui n'ont pas de frontière\nC'est l'ardente prière de la nuit des gitans\nOù vas-tu gitan?\nJe vais en Bohème\nOù vas-tu gitan?\nRevoir l'Italie\nEt toi beau gitan?\nEn Andalousie\nEt toi vieux gitan mon ami?\nJe suis bien trop vieux, moi je reste ici...\n\nAvant de repartir pour un nouveau voyage\nVers d'autres paysages\nSur des chemins mouvants\nLaisse encor un instant vagabonder ton rêve\nAvant que la nuit brève\nLe réduise à néant\nChante, gi